In [1]:
import requests
import json
import time
from pathlib import Path
from tqdm.auto import tqdm

PROJECT_ROOT = Path.home() / "xai-knowledge-graph"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = RAW_DIR / "arxiv_xai_papers.json"
OUTPUT_PATH = PROCESSED_DIR / "papers_enriched.json"

S2_BATCH_URL = "https://api.semanticscholar.org/graph/v1/paper/batch"
S2_FIELDS = (
    "title,venue,year,citationCount,"
    "references.paperId,references.externalIds,references.title,"
    "authors.authorId,authors.name,externalIds"
)

with open(INPUT_PATH) as f:
    papers = json.load(f)
print(f"Loaded {len(papers)} arXiv papers")

Loaded 3913 arXiv papers


In [2]:
def fetch_s2_batch(arxiv_ids, fields, batch_size=500):
    results = []
    for i in tqdm(range(0, len(arxiv_ids), batch_size), desc="S2 batches"):
        batch = [f"ARXIV:{aid}" for aid in arxiv_ids[i:i+batch_size]]
        try:
            r = requests.post(
                S2_BATCH_URL,
                params={"fields": fields},
                json={"ids": batch},
                timeout=60,
            )
            if r.status_code == 200:
                results.extend(r.json())
            else:
                print(f" Batch {i}: HTTP {r.status_code}")
                results.extend([None] * len(batch))
        except Exception as e:
            print(f" Batch {i} error: {e}")
            results.extend([None] * len(batch))
        time.sleep(1.0)
    return results

arxiv_ids = [p["arxiv_id"] for p in papers]
s2_data = fetch_s2_batch(arxiv_ids, S2_FIELDS)
matched = sum(1 for x in s2_data if x is not None)
print(f"\n Matched {matched}/{len(s2_data)} ({matched/len(s2_data)*100:.1f}%)")

S2 batches:   0%|          | 0/8 [00:00<?, ?it/s]

 Batch 1000: HTTP 429

 Matched 3379/3913 (86.4%)


In [3]:
def fetch_s2_batch_with_retry(arxiv_ids, fields, batch_size=200, max_retries=5):
    """Smaller batches + exponential backoff on 429."""
    results = []
    for i in tqdm(range(0, len(arxiv_ids), batch_size), desc="Retry batches"):
        batch = [f"ARXIV:{aid}" for aid in arxiv_ids[i:i+batch_size]]
        success = False
        for attempt in range(max_retries):
            try:
                r = requests.post(
                    S2_BATCH_URL,
                    params={"fields": fields},
                    json={"ids": batch},
                    timeout=60,
                )
                if r.status_code == 200:
                    results.extend(r.json())
                    success = True
                    break
                elif r.status_code == 429:
                    wait = (2 ** attempt) * 5   # 5, 10, 20, 40, 80 seconds
                    print(f"  Batch starting at {i}: 429, waiting {wait}s (attempt {attempt+1}/{max_retries})")
                    time.sleep(wait)
                else:
                    print(f"  Batch starting at {i}: HTTP {r.status_code}")
                    break
            except Exception as e:
                print(f"  Batch starting at {i}: {e}")
                time.sleep(5)
        if not success:
            results.extend([None] * len(batch))
        time.sleep(3.0)   # polite base delay between batches
    return results

# Find which papers we're still missing
missing_indices = [i for i, x in enumerate(s2_data) if x is None]
missing_arxiv_ids = [arxiv_ids[i] for i in missing_indices]
print(f"Re-fetching {len(missing_indices)} missing papers in smaller batches with backoff...\n")

recovered = fetch_s2_batch_with_retry(missing_arxiv_ids, S2_FIELDS, batch_size=200)

# Update s2_data in place
for orig_idx, new_data in zip(missing_indices, recovered):
    if new_data is not None:
        s2_data[orig_idx] = new_data

# Re-count
matched = sum(1 for x in s2_data if x is not None)
print(f"\n Now matched {matched}/{len(s2_data)} ({matched/len(s2_data)*100:.1f}%)")

Re-fetching 534 missing papers in smaller batches with backoff...



Retry batches:   0%|          | 0/3 [00:00<?, ?it/s]

  Batch starting at 200: 429, waiting 5s (attempt 1/5)

 Now matched 3878/3913 (99.1%)


In [4]:
enriched = []
for ap, sp in zip(papers, s2_data):
    p = dict(ap)
    if sp:
        p["s2_paper_id"]   = sp.get("paperId")
        p["venue"]         = sp.get("venue") or None
        p["citation_count"] = sp.get("citationCount", 0) or 0
        refs = sp.get("references") or []
        p["references_s2_ids"] = [r["paperId"] for r in refs if r and r.get("paperId")]
        p["references_arxiv_ids"] = [
            r["externalIds"]["ArXiv"]
            for r in refs
            if r and r.get("externalIds") and r["externalIds"].get("ArXiv")
        ]
        p["s2_authors"] = [
            {"author_id": a.get("authorId"), "name": a.get("name")}
            for a in (sp.get("authors") or [])
        ]
    else:
        p["s2_paper_id"], p["venue"], p["citation_count"] = None, None, 0
        p["references_s2_ids"], p["references_arxiv_ids"], p["s2_authors"] = [], [], []
    enriched.append(p)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(enriched, f, indent=2, ensure_ascii=False)
print(f"Saved → {OUTPUT_PATH}")

Saved → /slurm/homes/pchandra/xai-knowledge-graph/data/processed/papers_enriched.json


In [5]:
import pandas as pd
df = pd.DataFrame(enriched)
print(f"With venue: {df['venue'].notna().sum()} ({df['venue'].notna().mean()*100:.0f}%)")
print(f"Total citations: {df['citation_count'].sum():,}")
print(f"Median citations: {df['citation_count'].median():.0f}")
print(f"Avg references per paper: {df['references_s2_ids'].apply(len).mean():.1f}")
print(f"\nTop 5 cited papers:")
for _, row in df.nlargest(5, 'citation_count').iterrows():
    print(f"  [{row['citation_count']:>6}] {row['title'][:70]}")
print(f"\nTop venues:")
print(df['venue'].value_counts().head(10).to_string())

With venue: 3305 (84%)
Total citations: 83,709
Median citations: 1
Avg references per paper: 20.2

Top 5 cited papers:
  [ 26975] Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Lo
  [  5264] Explanation in Artificial Intelligence: Insights from the Social Scien
  [  3132] Grad-CAM++: Improved Visual Explanations for Deep Convolutional Networ
  [  1817] Consistent Individualized Feature Attribution for Tree Ensembles
  [  1089] Fooling LIME and SHAP: Adversarial Attacks on Post hoc Explanation Met

Top venues:
venue
arXiv.org                                                         1737
xAI                                                                 56
AAAI Conference on Artificial Intelligence                          53
Computer Vision and Pattern Recognition                             23
International Conference on Machine Learning                        23
Neural Information Processing Systems                               21
IEEE International Joint Confere